## Práctico 5: Volcano Crossing

In [ ]:
from volcano_crossing_env import VolcanoCrossing

env = VolcanoCrossing(slip_prob=0.5)

In [ ]:
env.reset()

### Jugar manualmente

In [ ]:
def run_game():
        print("Play manually...")
        obs = env.reset()
        print(obs)
        done = False
        step_counter = 0
        all_rewards = 0
        env.render()

        while not done:
            action = input("Next action: ")
            env.check_action(action)
            obs, reward, done_env, _ = env.step(action)
            print(f'{obs=} {reward=} {done_env=}')
            all_rewards += reward
            done = done_env
            env.render()
            step_counter += 1

In [ ]:
#run_game()

### Jugar con una policy

In [ ]:
def policy_south(state) :
  return 'S'

In [ ]:
def run(policy) :
  U = 0
  done = False
  state = env.reset()
  while not done:
    state, reward, done, _ = env.step(policy(state))
    U += reward
  return U

In [ ]:
run(policy_south)

### Estimación de la policy por promedio

In [ ]:
r = 0
N = 50_000
for i in range(N):
    r += run(policy_south)

print (f'Average reward: {r/N}')

### Policy Evaluation

In [ ]:
def policy_evaluation(policy, delta = 0.05, gamma = 0.999):
  V = { '11':0, '12':0, '13':0, '14':0, '21':0, '22':0, '23':0, '24':0, '31':0, '32':0, '33':0, '34':0 }
  V_new = { '11':0, '12':0, '13':0, '14':0, '21':0, '22':0, '23':0, '24':0, '31':0, '32':0, '33':0, '34':0 }
  max_diff = delta + 1

  while max_diff > delta:
    for s in env.observation_space:
      a = policy(s)
      q = 0
      for s_prime in env.P[s][a].keys():
        q += env.P[s][a][s_prime] * (env.R[s][a][s_prime] + gamma * V[s_prime])
      V_new[s] = q
    max_diff = 0
    for s in V.keys():
      max_diff = max(max_diff, abs(V[s]-V_new[s]))
    V = V_new.copy()
  return V

In [ ]:
V = policy_evaluation(policy_south, delta = 0.05, gamma = 0.999)
print(V)

### Policy Improvement

In [ ]:
policies = { '11':'S', '12':'S', '13':'S', '14':'S', '21':'S', '22':'S', '23':'S', '24':'S', '31':'S', '32':'S', '33':'S', '34':'S' }

In [ ]:
def policy_improvement_step(V, gamma=0.999):
  new_policy = {}
  for s in env.observation_space:
    best_action = None
    best_q = float('-inf')
    for a in env.action_space:
      q = 0
      for s_prime in env.P[s][a].keys():
        q += env.P[s][a][s_prime] * (env.R[s][a][s_prime] + gamma * V[s_prime])
      if q > best_q:
        best_q = q
        best_action = a
    new_policy[s] = best_action
  return new_policy

V = policy_evaluation(policy_south, delta=0.05, gamma=0.999)
improved_policy = policy_improvement_step(V, gamma=0.999)
print('Política inicial (todo Sur):', policies)
print('Política mejorada:         ', improved_policy)

### Policy Iteration

In [ ]:
def policy_iteration(delta=0.05, gamma=0.999):
  # Política inicial arbitraria: todo Sur
  policy_dict = {s: 'S' for s in env.observation_space}
  pi_history = []  # V['21'] por iteración (para graficar)

  def current_policy(state):
    return policy_dict[state]

  while True:
    # Paso 1: Policy Evaluation
    V = policy_evaluation(current_policy, delta=delta, gamma=gamma)
    pi_history.append(V['21'])

    # Paso 2: Policy Improvement
    policy_stable = True
    for s in env.observation_space:
      old_action = policy_dict[s]
      best_action, best_q = None, float('-inf')
      for a in env.action_space:
        q = sum(
          env.P[s][a][sp] * (env.R[s][a][sp] + gamma * V[sp])
          for sp in env.P[s][a].keys()
        )
        if q > best_q:
          best_q, best_action = q, a
      policy_dict[s] = best_action
      if old_action != best_action:
        policy_stable = False

    if policy_stable:
      break

  return policy_dict, V, pi_history

opt_policy, opt_V, pi_history = policy_iteration()
print('Política óptima (Policy Iteration):', opt_policy)
print('Función de valor óptima:', opt_V)
print('V("21") por iteración:', pi_history)

### Value Iteration

1. Implementar el algoritmo de Value Iteration para encontrar la policy óptima.
2. Obtener la utilidad esperada de esa policy.
3. Graficar el valor del estado "21" para cada iteración en Value Iteration y Policy Iteration (con una policy inicial arbitraria).

In [ ]:
def q_value(mdp, state, action, V, gamma):
  return sum(
    mdp.P[state][action][sp] * (mdp.R[state][action][sp] + gamma * V[sp])
    for sp in mdp.P[state][action].keys()
  )

def value_iteration(mdp, delta_threshold=0.05, gamma=0.999):
  V = {s: 0 for s in mdp.P.keys()}
  vi_history = []  # V['21'] por iteración

  while True:
    V_new = V.copy()
    for s in mdp.P.keys():
      V_new[s] = max(
        q_value(mdp, s, a, V, gamma)
        for a in mdp.action_space
      )
    max_diff = max(abs(V[s] - V_new[s]) for s in V.keys())
    V = V_new
    vi_history.append(V['21'])
    if max_diff < delta_threshold:
      break

  optimal_policy = {
    s: max(mdp.action_space, key=lambda a: q_value(mdp, s, a, V, gamma))
    for s in mdp.P.keys()
  }
  return V, optimal_policy, vi_history

V_opt, vi_policy, vi_history = value_iteration(env)
print('Función de valor óptima (VI):', V_opt)
print('Política óptima (VI):', vi_policy)

In [ ]:
# Utilidad esperada de la política óptima simulando 50.000 episodios
def optimal_policy_fn(state):
  return vi_policy[state]

N = 50_000
r = sum(run(optimal_policy_fn) for _ in range(N))
print(f'Utilidad esperada (política óptima): {r / N:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(range(1, len(vi_history) + 1), vi_history,
        marker='o', label='Value Iteration')
ax.plot(range(1, len(pi_history) + 1), pi_history,
        marker='s', label='Policy Iteration')

ax.set_xlabel('Iteración')
ax.set_ylabel('V(estado "21")')
ax.set_title('Convergencia de V("21"): Value Iteration vs Policy Iteration')
ax.legend()
plt.tight_layout()
plt.show()